# Lesson 12A: Autoencoders - Theory

Learn neural network-based dimensionality reduction.

**Autoencoder** is an unsupervised neural network that learns to compress and reconstruct data.

**Architecture:**
- **Encoder**: Compresses input to latent representation z
- **Latent space**: Low-dimensional bottleneck
- **Decoder**: Reconstructs input from z

**Loss**: Reconstruction error (e.g., MSE between input and output)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

## Mathematics

For input x and latent z:

**Encoder**: z = f_θ(x)  
**Decoder**: x̂ = g_φ(z)  
**Loss**: L = ||x - x̂||²

Training minimizes reconstruction error, forcing the network to learn efficient representations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_digits

# Simple Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim=64, latent_dim=8):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

# Load data
digits = load_digits()
X = digits.data / 16.0  # Normalize to [0, 1]
X_tensor = torch.FloatTensor(X)

# Train autoencoder
model = Autoencoder(input_dim=64, latent_dim=8)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(50):
    x_recon, z = model(X_tensor)
    loss = criterion(x_recon, X_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50, Loss: {loss.item():.4f}")

# Visualize
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
with torch.no_grad():
    x_recon, _ = model(X_tensor[:5])
for i in range(5):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original')
    axes[1, i].imshow(x_recon[i].numpy().reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Reconstructed')
plt.tight_layout()
plt.show()
print("✅ Autoencoder trained! 64D → 8D → 64D")